In [2]:
import numpy as np
import pandas as pd
import arff
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from autofeat import AutoFeatRegressor 
import featuretools as ft


In [24]:
with open("german-credit-R.arff", "r") as file:
    data = arff.load(file)
df = pd.DataFrame(data["data"], columns=[attr[0] for attr in data["attributes"]])
 

In [25]:
cat_cols = ["Sex", "Housing", "Saving accounts", "Checking account", "Purpose","Risk"]
for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

In [37]:
df.isna().sum()

Age                 0
Sex                 0
Job                 0
Housing             0
Saving accounts     0
Checking account    0
Credit amount       0
Duration            0
Purpose             0
Risk                0
customer_id         0
dtype: int64

In [5]:
X = df.drop(columns=["Risk"])  
y = df["Risk"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Précision : {accuracy_score(y_test, y_pred):.2f}")
print(classification_report(y_test, y_pred))
print("Matrice de confusion :\n", confusion_matrix(y_test, y_pred))

Précision : 0.76
              precision    recall  f1-score   support

           0       0.65      0.37      0.47        59
           1       0.78      0.91      0.84       141

    accuracy                           0.76       200
   macro avg       0.71      0.64      0.66       200
weighted avg       0.74      0.76      0.73       200

Matrice de confusion :
 [[ 22  37]
 [ 12 129]]


In [12]:
#https://medium.com/@boukamchahamdi/autofeat-automating-feature-engineering-with-python-f22ec23265a9
af = AutoFeatRegressor( feateng_steps=2,n_jobs=-1)  

X_train_af = af.fit_transform(X_train, y_train)
X_test_af = af.transform(X_test)
X_train_af.head()
print(f"Nombre de nouvelles features créées : {X_train_af.shape[1] - X_train.shape[1]}")

model_af = LogisticRegression()
model_af.fit(X_train_af, y_train)
y_pred_af = model_af.predict(X_test_af)

print("\n Performance APRÈS AutoFeat")
print(f"Précision : {accuracy_score(y_test, y_pred_af):.2f}")
print(classification_report(y_test, y_pred_af))
print("Matrice de confusion :\n", confusion_matrix(y_test, y_pred_af))

c:\Users\H P\git\projetTutore\venv\lib\site-packages\autofeat\featsel.py:270: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  if np.max(np.abs(correlations[c].ravel()[:i])) < 0.9:
c:\Users\H P\git\projetTutore\venv\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Nombre de nouvelles features créées : 13

 Performance APRÈS AutoFeat
Précision : 0.77
              precision    recall  f1-score   support

           0       0.69      0.37      0.48        59
           1       0.78      0.93      0.85       141

    accuracy                           0.77       200
   macro avg       0.73      0.65      0.67       200
weighted avg       0.75      0.77      0.74       200

Matrice de confusion :
 [[ 22  37]
 [ 10 131]]


In [ ]:
to install
python -m pip install jupyter-server-proxy
distributed

In [51]:
df["customer_id"] = range(1, len(df) + 1)
customers = df[["customer_id", "Age", "Sex", "Job", "Housing", "Saving accounts", "Checking account"]]
loans = df[["customer_id", "Credit amount", "Duration", "Purpose", "Risk"]]
loans.reset_index()

,index,customer_id,Credit amount,Duration,Purpose,Risk
0,0,1,1169,6,5,1
1,1,2,5951,48,5,0
2,2,3,2096,12,3,1
3,3,4,7882,42,4,1
4,4,5,4870,24,1,0
...,...,...,...,...,...,...
995,995,996,1736,12,4,1
996,996,997,3857,30,1,1
997,997,998,804,12,5,1
998,998,999,1845,45,5,0


In [48]:
loans.index()

TypeError: 'Index' object is not callable

In [52]:


entities={
    "customers":(customers,"customer_id"),
    "loans":(loans,"index",)
}
relationships=[("customers","customer_id","loans","customer_id")
    
]

feature_matrix_customers, features_defs = ft.dfs(dataframes=entities,
                                                 relationships=relationships,
                                                 target_dataframe_name="customers")

c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\featuretools\entityset\entityset.py:1733: UserWarning: index index not found in dataframe, creating new integer column
  warnings.warn(
c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\featuretools\computational_backends\feature_set_calculator.py:785: FutureWarning: The provided callable <function std at 0x000001EBF2E971C0> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  ).agg(to_agg)
c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\featuretools\computational_backends\feature_set_calculator.py:785: FutureWarning: The provided callable <function mean at 0x000001EBF2E970A0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  ).agg(to_agg)
c:\Users\H P\git\projetTutoreA

In [53]:
feature_matrix_customers

,Age,Sex,Job,Housing,Saving accounts,Checking account,COUNT(loans),MAX(loans.Credit amount),MAX(loans.Duration),MAX(loans.Purpose),...,SKEW(loans.Purpose),SKEW(loans.Risk),STD(loans.Credit amount),STD(loans.Duration),STD(loans.Purpose),STD(loans.Risk),SUM(loans.Credit amount),SUM(loans.Duration),SUM(loans.Purpose),SUM(loans.Risk)
customer_id,,,,,,,,,,,,,,,,,,,,,
1,67,1,2,1,4,0,1,1169.0,6.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1169.0,6.0,5.0,1.0
2,22,0,2,1,0,1,1,5951.0,48.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,5951.0,48.0,5.0,0.0
3,49,1,1,1,0,3,1,2096.0,12.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2096.0,12.0,3.0,1.0
4,45,1,2,0,0,0,1,7882.0,42.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,7882.0,42.0,4.0,1.0
5,53,1,2,0,0,0,1,4870.0,24.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,4870.0,24.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,31,0,1,1,0,3,1,1736.0,12.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1736.0,12.0,4.0,1.0
997,40,1,3,1,0,0,1,3857.0,30.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,3857.0,30.0,1.0,1.0
998,38,1,2,1,0,3,1,804.0,12.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,804.0,12.0,5.0,1.0


In [28]:
es = ft.EntitySet(id="credit_risk")

In [30]:
es = es.add_dataframe(dataframe_name="customers", dataframe=customers, index="customer_id")
es = es.add_dataframe(dataframe_name="loans", dataframe=loans,make_index = True, index="custo_loan_id")


c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\featuretools\entityset\entityset.py:724: UserWarning: A Woodwork-initialized DataFrame was provided, so the following parameters were ignored: index
  warnings.warn(


In [31]:
es = es.add_relationship("customers","customer_id","loans","customer_id")


In [32]:
print(es)

Entityset: credit_risk
  DataFrames:
    customers [Rows: 1000, Columns: 7]
    loans [Rows: 1000, Columns: 6]
  Relationships:
    loans.customer_id -> customers.customer_id


In [33]:
agg_primitives =  ["sum", "std", "max", "skew", "min", "mean", "count", "percent_true", "num_unique", "mode"]
trans_primitives =  ["day", "year", "month", "weekday", "haversine", 
"num_words", "num_characters"]

In [38]:
print(trans_primitives)
print(agg_primitives)

['day', 'year', 'month', 'weekday', 'haversine', 'num_words', 'num_characters']
['sum', 'std', 'max', 'skew', 'min', 'mean', 'count', 'percent_true', 'num_unique', 'mode']


In [39]:
feature_matrix, feature_defs = ft.dfs(entityset = es, 
target_dataframe_name = 'loans',
    trans_primitives = trans_primitives,
    agg_primitives=agg_primitives,
    max_depth = 2, n_jobs = -1, verbose = 1)

c:\Users\H P\git\projetTutoreAutoFE\venv\lib\site-packages\featuretools\synthesis\dfs.py:321: UnusedPrimitiveWarning: Some specified primitives were not used during DFS:
  trans_primitives: ['day', 'haversine', 'month', 'num_characters', 'num_words', 'weekday', 'year']
  agg_primitives: ['mode', 'num_unique', 'percent_true']
This may be caused by a using a value of max_depth that is too small, not setting interesting values, or it may indicate no compatible columns for the primitive were found in the data. If the DFS call contained multiple instances of a primitive in the list above, none of them were used.
  warnings.warn(warning_msg, UnusedPrimitiveWarning)


Built 36 features
Elapsed: 00:00 | Progress:   0%|          

2025-03-25 11:18:04,923 INFO: State start
2025-03-25 11:18:04,955 INFO:   Scheduler at:     tcp://127.0.0.1:63003
2025-03-25 11:18:04,955 INFO:   dashboard at:  http://127.0.0.1:8787/status
2025-03-25 11:18:04,955 INFO: Registering Worker plugin shuffle
2025-03-25 11:18:05,156 INFO:         Start Nanny at: 'tcp://127.0.0.1:63008'
2025-03-25 11:18:05,156 INFO:         Start Nanny at: 'tcp://127.0.0.1:63012'
2025-03-25 11:18:05,156 INFO:         Start Nanny at: 'tcp://127.0.0.1:63014'
2025-03-25 11:18:05,178 INFO:         Start Nanny at: 'tcp://127.0.0.1:63013'
2025-03-25 11:18:05,192 INFO:         Start Nanny at: 'tcp://127.0.0.1:63006'
2025-03-25 11:18:05,206 INFO:         Start Nanny at: 'tcp://127.0.0.1:63017'
2025-03-25 11:18:05,227 INFO:         Start Nanny at: 'tcp://127.0.0.1:63010'
2025-03-25 11:18:05,240 INFO:         Start Nanny at: 'tcp://127.0.0.1:63009'
2025-03-25 11:18:05,261 INFO:         Start Nanny at: 'tcp://127.0.0.1:63011'
2025-03-25 11:18:05,276 INFO:         Start 

Elapsed: 00:18 | Progress:   0%|          


AttributeError: 'NoneType' object has no attribute 'status'

In [ ]:
#https://www.analyticsvidhya.com/blog/2018/08/guide-automated-feature-engineering-featuretools-python/
#https://ranasinghiitkgp.medium.com/feature-engineering-using-featuretools-with-code-10f8c83e5f68
#Avec featuretools

df_ft = df.reset_index().rename(columns={'index': 'id'})
es = ft.EntitySet(id="dataset")

es = es.add_dataframe(
    dataframe_name="df_ft",  
    dataframe=df_ft,
    index="id"  
)

#création d'une entité (table) job
es.normalize_entity(
    base_dataframe_name="df_ft",
    new_dataframe_name="job_table",
    index="Job"
)

es.normalize_entity(
    base_dataframe_name="df_ft",
    new_dataframe_name="housing_table",
    index="Housing"
)

es.normalize_entity(
    base_dataframe_name="df_ft",
    new_dataframe_name="purpose_table",
    index="Purpose"
)

feature_matrix, feature_names = ft.dfs(
    entityset=es, 
    target_entity ="df_ft",  
    max_depth=4,
    
)

print(f"Nombre de nouvelles features créées : {feature_matrix.shape[1] - X.shape[1]}")



Nombre de nouvelles features créées : 1


c:\Users\H P\git\projetTutore\venv\lib\site-packages\featuretools\synthesis\deep_feature_synthesis.py:169: UserWarning: Only one dataframe in entityset, changing max_depth to 1 since deeper features cannot be created
  warnings.warn(


In [22]:
print(es)

Entityset: dataset
  DataFrames:
    df_ft [Rows: 1000, Columns: 11]
  Relationships:
    No relationships


In [ ]:
X_train_ft, X_test_ft, y_train_ft, y_test_ft = train_test_split(feature_matrix, y, test_size=0.2, random_state=42)
model = LogisticRegression()
model.fit(X_train_ft, y_train_ft)
y_pred_ft = model.predict(X_test_ft)

print("\n Performance APRÈS AutoFeat")
print(f"Précision : {accuracy_score(y_test_ft, y_pred_ft):.2f}")
print(classification_report(y_test_ft, y_pred_ft))
print("Matrice de confusion :\n", confusion_matrix(y_test_ft, y_pred_ft))
